# 2.3 The Inverse of a Matrix

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 2 — Matrices**

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import add, scalar_mul, multiply, transpose, inverse
from linalg_core.elimination import to_rref, solve
from linalg_core import EPSILON

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def matrices_equal(A, B):
    """Check if two matrices are equal within EPSILON tolerance."""
    if A.rows != B.rows or A.cols != B.cols:
        return False
    return all(abs(a - b) < EPSILON for a, b in zip(A.data, B.data))


def to_np(mat):
    """Convert our Matrix to a NumPy array."""
    return np.array(mat.to_lists())


def check_match(ours, theirs, label):
    """Compare our Matrix result against a NumPy array."""
    ours_np = to_np(ours)
    match = np.allclose(ours_np, theirs, atol=1e-9)
    status = "PASS" if match else "FAIL"
    print(f"[{status}] {label}")
    if not match:
        print(f"  Ours:   {ours_np}")
        print(f"  NumPy:  {theirs}")
    return match


print("Setup complete.")

---

## 2. Motivation — Cracking a Code

A simple **encryption scheme** works like this: encode a message as a vector
$\mathbf{x}$ of integers, then multiply by a secret **key matrix** $A$ to produce
the ciphertext $\mathbf{y} = A\mathbf{x}$.

To **decode**, the receiver needs to recover $\mathbf{x}$ from $\mathbf{y}$.
If $A$ has an inverse $A^{-1}$, then:

$$\mathbf{x} = A^{-1}\mathbf{y}$$

Let's try it. Suppose the key matrix and ciphertext are:

$$A = \begin{bmatrix} 1 & 2 \\ 3 & 7 \end{bmatrix}, \quad
  \mathbf{y} = \begin{bmatrix} 11 \\ 31 \end{bmatrix}$$

What is the original message $\mathbf{x}$? We'll answer this by the end of the
Build section, once we know how to compute $A^{-1}$.

In [ ]:
A_key = Matrix.from_lists([[1, 2], [3, 7]])
y_cipher = Matrix.from_vector([11, 31])

print("Key matrix A =")
print(A_key)
print("\nCiphertext y =")
print(y_cipher)
print("\nGoal: find x such that A * x = y.")
print("That is, x = A^{-1} * y.")

---

## 3. Build — The Inverse of a Matrix

We develop the theory and computation of matrix inverses step by step:
first the definition, then the $2 \times 2$ formula, then the general
Gauss–Jordan method, and finally how inverses help solve systems.

### 3.1 Definition — Invertible Matrices

> **Definition (Larson, Sec. 2.3).** An $n \times n$ matrix $A$ is **invertible**
> (or **nonsingular**) if there exists an $n \times n$ matrix $B$ such that
>
> $$AB = BA = I_n$$
>
> The matrix $B$ is called the **inverse** of $A$ and is denoted $A^{-1}$.
> A matrix that is not invertible is called **singular**.

**Key facts:**
- Only **square** matrices can be invertible.
- If $A^{-1}$ exists, it is **unique** (Larson, Theorem 2.7).
- *Proof of uniqueness:* If $B$ and $C$ are both inverses of $A$, then
  $B = BI = B(AC) = (BA)C = IC = C$.

Let's verify the definition with a known example.

In [ ]:
A = Matrix.from_lists([[1, 2], [3, 7]])
B = Matrix.from_lists([[7, -2], [-3, 1]])
I2 = Matrix.identity(2)

AB = multiply(A, B)
BA = multiply(B, A)

print("A =")
print(A)
print("\nB =")
print(B)
print("\nAB =")
print(AB)
print("\nBA =")
print(BA)

print(f"\nAB == I? {matrices_equal(AB, I2)}")
print(f"BA == I? {matrices_equal(BA, I2)}")
print("\nSince AB = BA = I, B is the inverse of A.")

### 3.2 The $2 \times 2$ Inverse Formula

For a $2 \times 2$ matrix

$$A = \begin{bmatrix} a & b \\ c & d \end{bmatrix}$$

there is a direct formula (Larson, Theorem 2.8):

$$A^{-1} = \frac{1}{ad - bc} \begin{bmatrix} d & -b \\ -c & a \end{bmatrix}$$

provided the **determinant** $\det(A) = ad - bc \neq 0$.

If $\det(A) = 0$, the matrix is **singular** and has no inverse.

Let's implement this formula and test it.

In [ ]:
def inverse_2x2(A):
    """Compute the inverse of a 2x2 matrix using the direct formula.

    A^{-1} = (1/det) * [[d, -b], [-c, a]]
    Raises ValueError if det(A) = 0.
    """
    if A.rows != 2 or A.cols != 2:
        raise ValueError("Only 2x2 matrices supported")
    a, b = A[0, 0], A[0, 1]
    c, d = A[1, 0], A[1, 1]
    det = a * d - b * c
    if abs(det) < EPSILON:
        raise ValueError(f"Matrix is singular (det = {det})")
    return Matrix.from_lists([
        [ d / det, -b / det],
        [-c / det,  a / det],
    ])


A = Matrix.from_lists([[1, 2], [3, 7]])
A_inv = inverse_2x2(A)

print("A =")
print(A)
print(f"\ndet(A) = (1)(7) - (2)(3) = {1*7 - 2*3}")
print("\nA^{-1} (via 2x2 formula) =")
print(A_inv)

print("\nVerification:")
print("A * A^{-1} =")
print(multiply(A, A_inv))
print(f"\nEquals I? {matrices_equal(multiply(A, A_inv), Matrix.identity(2))}")

### 3.3 Decode the Message

Now we can solve the encryption problem from the motivation.

Given $A = \begin{bmatrix} 1 & 2 \\ 3 & 7 \end{bmatrix}$ and
$\mathbf{y} = \begin{bmatrix} 11 \\ 31 \end{bmatrix}$, compute:

$$\mathbf{x} = A^{-1}\mathbf{y}$$

In [ ]:
A_inv_key = inverse_2x2(A_key)

print("A^{-1} =")
print(A_inv_key)

x_decoded = multiply(A_inv_key, y_cipher)
print("\nDecoded message x = A^{-1} * y =")
print(x_decoded)
print(f"\nx = [{x_decoded[0,0]:.0f}, {x_decoded[1,0]:.0f}]")

print("\n--- Verification ---")
y_check = multiply(A_key, x_decoded)
print("A * x =")
print(y_check)
print(f"\nA * x == y? {matrices_equal(y_check, y_cipher)}")

print("\n--- Double-check: A * A^{-1} = I ---")
product = multiply(A_key, A_inv_key)
print("A * A^{-1} =")
print(product)
print(f"Equals I? {matrices_equal(product, Matrix.identity(2))}")

### 3.4 Gauss–Jordan Method for Finding $A^{-1}$

The $2 \times 2$ formula is neat but doesn't generalize easily. The **Gauss–Jordan
method** works for any $n \times n$ matrix:

1. Form the **augmented matrix** $[A \mid I_n]$.
2. Apply row operations to reduce the left half to $I_n$.
3. The right half becomes $A^{-1}$.

$$[A \mid I] \xrightarrow{\text{RREF}} [I \mid A^{-1}]$$

If the left side cannot be reduced to $I$ (i.e., $A$ is singular), the
process reveals this by producing a row of zeros in the left half.

Let's watch this happen step by step with `to_rref(verbose=True)`.

In [ ]:
A = Matrix.from_lists([[1, 2], [3, 7]])
n = A.rows

aug = Matrix(n, 2 * n)
for i in range(n):
    for j in range(n):
        aug[i, j] = A[i, j]
    aug[i, n + i] = 1.0

print("Augmented matrix [A | I]:")
print(aug)
print("\n" + "=" * 50)
print("Row reduction steps:")
print("=" * 50)

to_rref(aug, verbose=True)

print("\nFinal RREF [I | A^{-1}]:")
print(aug)

A_inv_gj = Matrix(n, n)
for i in range(n):
    for j in range(n):
        A_inv_gj[i, j] = aug[i, n + j]

print("\nA^{-1} (extracted from right half) =")
print(A_inv_gj)

print(f"\nMatches 2x2 formula result? {matrices_equal(A_inv_gj, A_inv_key)}")

Let's also demonstrate with a $3 \times 3$ matrix, where the $2 \times 2$
formula no longer applies and Gauss–Jordan is essential.

In [ ]:
A3 = Matrix.from_lists([
    [1, 0, 2],
    [2, -1, 3],
    [4, 1, 8],
])

n = A3.rows
aug3 = Matrix(n, 2 * n)
for i in range(n):
    for j in range(n):
        aug3[i, j] = A3[i, j]
    aug3[i, n + i] = 1.0

print("A (3x3) =")
print(A3)
print("\nAugmented [A | I]:")
print(aug3)
print("\n" + "=" * 50)
print("Row reduction steps:")
print("=" * 50)

to_rref(aug3, verbose=True)

print("\nFinal RREF:")
print(aug3)

A3_inv = Matrix(n, n)
for i in range(n):
    for j in range(n):
        A3_inv[i, j] = aug3[i, n + j]

print("\nA^{-1} =")
print(A3_inv)

print("\nVerification: A * A^{-1} =")
print(multiply(A3, A3_inv))
print(f"\nEquals I? {matrices_equal(multiply(A3, A3_inv), Matrix.identity(3))}")

### 3.5 Using `inverse(A)` from `linalg_core.ops`

The Gauss–Jordan procedure is already packaged in `linalg_core.ops.inverse`.
It augments $[A \mid I]$, calls `to_rref`, checks for singularity, and
extracts the right half. Let's demo it.

In [ ]:
A = Matrix.from_lists([[1, 2], [3, 7]])
A_inv = inverse(A)

print("A =")
print(A)
print("\ninverse(A) =")
print(A_inv)
print(f"\nA * A^{{-1}} == I? {matrices_equal(multiply(A, A_inv), Matrix.identity(2))}")

print("\n" + "-" * 40)

A3 = Matrix.from_lists([
    [1, 0, 2],
    [2, -1, 3],
    [4, 1, 8],
])
A3_inv = inverse(A3)

print("\nA3 =")
print(A3)
print("\ninverse(A3) =")
print(A3_inv)
print(f"\nA3 * A3^{{-1}} == I? {matrices_equal(multiply(A3, A3_inv), Matrix.identity(3))}")
print(f"A3^{{-1}} * A3 == I? {matrices_equal(multiply(A3_inv, A3), Matrix.identity(3))}")

### 3.6 Solving Systems via the Inverse

If $A$ is invertible, the system $A\mathbf{x} = \mathbf{b}$ has the unique solution

$$\mathbf{x} = A^{-1}\mathbf{b}$$

This is elegant but **not always efficient**. For a single right-hand side,
Gaussian elimination is faster ($O(n^3)$ either way, but computing the full
inverse does extra work). However, the inverse shines when you need to solve
the **same system with multiple right-hand sides**.

Let's compare both approaches on a concrete system.

In [ ]:
A = Matrix.from_lists([
    [1, 0, 2],
    [2, -1, 3],
    [4, 1, 8],
])
b = Matrix.from_vector([1, 1, 3])

print("System: Ax = b")
print("A ="); print(A)
print("\nb ="); print(b)

print("\n--- Method 1: x = A^{-1} * b ---")
A_inv = inverse(A)
x_inv = multiply(A_inv, b)
print("x ="); print(x_inv)

print("\n--- Method 2: RREF on [A | b] ---")
aug = Matrix(3, 4)
for i in range(3):
    for j in range(3):
        aug[i, j] = A[i, j]
    aug[i, 3] = b[i, 0]

sol_type, sol = solve(aug)
print(f"Solution type: {sol_type}")
print(f"x = {sol}")

print("\n--- Comparison ---")
x_rref = Matrix.from_vector(sol)
print(f"Both methods agree? {matrices_equal(x_inv, x_rref)}")

print("\n--- Verification: A * x = b ---")
print("A * x =")
print(multiply(A, x_inv))
print(f"\nA * x == b? {matrices_equal(multiply(A, x_inv), b)}")

### 3.7 When the Inverse Doesn't Exist

A matrix is **singular** (non-invertible) when its rows are linearly dependent.
Geometrically, a singular matrix "collapses" $n$-dimensional space into a
lower-dimensional subspace — information is lost, and the mapping cannot be
reversed.

In the Gauss–Jordan procedure, this manifests as a **row of zeros** appearing
in the left half of $[A \mid I]$ during reduction. The left side can never
become the identity, so no inverse exists.

**Example.** The matrix

$$A = \begin{bmatrix} 1 & 2 \\ 2 & 4 \end{bmatrix}$$

is singular because row 2 = 2 $\times$ row 1.

In [ ]:
A_sing = Matrix.from_lists([[1, 2], [2, 4]])

print("A =")
print(A_sing)
print(f"\ndet(A) = (1)(4) - (2)(2) = {1*4 - 2*2}")
print("Rows are linearly dependent: R2 = 2 * R1")

print("\n--- Attempting Gauss-Jordan on [A | I] ---")
n = A_sing.rows
aug = Matrix(n, 2 * n)
for i in range(n):
    for j in range(n):
        aug[i, j] = A_sing[i, j]
    aug[i, n + i] = 1.0

print("[A | I] =")
print(aug)

to_rref(aug, verbose=True)

print("\nAfter RREF:")
print(aug)
print("\nLeft half has a zero row => A is SINGULAR. No inverse exists.")

In [ ]:
print("--- Calling inverse() on a singular matrix ---")
try:
    inverse(A_sing)
except ValueError as e:
    print(f"Caught expected error: {e}")

print("\n--- Another singular example: 3x3 ---")
A_sing3 = Matrix.from_lists([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
])
print("A =")
print(A_sing3)
print("Note: R3 = 2*R2 - R1, so rows are linearly dependent.")

try:
    inverse(A_sing3)
except ValueError as e:
    print(f"\nCaught expected error: {e}")
    print("Confirmed: [[1,2,3],[4,5,6],[7,8,9]] is singular.")

---

## 4. Verify — Properties of the Inverse

If $A$ and $B$ are invertible $n \times n$ matrices, the inverse satisfies
several important algebraic properties (Larson, Theorem 2.9):

| Property | Statement |
|---|---|
| Left identity | $A^{-1}A = I$ |
| Right identity | $AA^{-1} = I$ |
| Involution | $(A^{-1})^{-1} = A$ |
| Product rule | $(AB)^{-1} = B^{-1}A^{-1}$ (order reverses!) |
| Scalar rule | $(cA)^{-1} = \frac{1}{c}A^{-1}$ for $c \neq 0$ |
| Transpose rule | $(A^T)^{-1} = (A^{-1})^T$ |

Let's verify each one computationally.

In [ ]:
A = Matrix.from_lists([[1, 2], [3, 7]])
B = Matrix.from_lists([[2, 1], [5, 3]])
c = 4.0
I2 = Matrix.identity(2)

A_inv = inverse(A)
B_inv = inverse(B)

print("A ="); print(A)
print("\nB ="); print(B)
print(f"\nc = {c}")

print("\n" + "=" * 50)
print("Property 1: A^{-1} * A = I")
ok1 = matrices_equal(multiply(A_inv, A), I2)
print(f"  A^{{-1}} * A == I? {ok1}")

print("\nProperty 2: A * A^{-1} = I")
ok2 = matrices_equal(multiply(A, A_inv), I2)
print(f"  A * A^{{-1}} == I? {ok2}")

print("\nProperty 3: (A^{-1})^{-1} = A")
ok3 = matrices_equal(inverse(A_inv), A)
print(f"  (A^{{-1}})^{{-1}} == A? {ok3}")

print("\nProperty 4: (AB)^{-1} = B^{-1} A^{-1}")
AB = multiply(A, B)
lhs4 = inverse(AB)
rhs4 = multiply(B_inv, A_inv)
ok4 = matrices_equal(lhs4, rhs4)
print(f"  (AB)^{{-1}} == B^{{-1}}A^{{-1}}? {ok4}")

print("\nProperty 5: (cA)^{-1} = (1/c) A^{-1}")
lhs5 = inverse(scalar_mul(A, c))
rhs5 = scalar_mul(A_inv, 1.0 / c)
ok5 = matrices_equal(lhs5, rhs5)
print(f"  ({c}A)^{{-1}} == (1/{c})A^{{-1}}? {ok5}")

print("\nProperty 6: (A^T)^{-1} = (A^{-1})^T")
lhs6 = inverse(transpose(A))
rhs6 = transpose(A_inv)
ok6 = matrices_equal(lhs6, rhs6)
print(f"  (A^T)^{{-1}} == (A^{{-1}})^T? {ok6}")

all_ok = all([ok1, ok2, ok3, ok4, ok5, ok6])
print(f"\nAll properties verified: {all_ok}")

### Cross-Check with NumPy

Our from-scratch inverse should match NumPy's `np.linalg.inv`.

In [ ]:
print("=" * 50)
print("NumPy cross-verification")
print("=" * 50)

A = Matrix.from_lists([[1, 2], [3, 7]])
A_np = to_np(A)
check_match(inverse(A), np.linalg.inv(A_np), "inverse(2x2) matches NumPy")

A3 = Matrix.from_lists([
    [1, 0, 2],
    [2, -1, 3],
    [4, 1, 8],
])
A3_np = to_np(A3)
check_match(inverse(A3), np.linalg.inv(A3_np), "inverse(3x3) matches NumPy")

A4 = Matrix.from_lists([
    [2, 1, 0, 3],
    [1, 0, 2, 1],
    [0, 3, 1, 2],
    [1, 2, 3, 0],
])
A4_np = to_np(A4)
check_match(inverse(A4), np.linalg.inv(A4_np), "inverse(4x4) matches NumPy")

print("\n--- Property: (AB)^{-1} = B^{-1} A^{-1} via NumPy ---")
B3 = Matrix.from_lists([
    [2, 0, 1],
    [0, 3, 2],
    [1, 1, 1],
])
B3_np = to_np(B3)
AB3 = multiply(A3, B3)
AB3_np = A3_np @ B3_np
check_match(
    inverse(AB3),
    np.linalg.inv(AB3_np),
    "inverse(A3*B3) matches NumPy"
)
check_match(
    inverse(AB3),
    np.linalg.inv(B3_np) @ np.linalg.inv(A3_np),
    "inverse(AB) == B^{-1}A^{-1} via NumPy"
)

---

## 5. Visualize — Inverse as an "Undo" Transformation

A $2 \times 2$ matrix $A$ defines a linear transformation on $\mathbb{R}^2$.
Its inverse $A^{-1}$ is the transformation that **undoes** $A$: if $A$ maps
the unit square to a parallelogram, then $A^{-1}$ maps that parallelogram
back to the unit square.

We visualize the **roundtrip**: Original $\xrightarrow{A}$ Transformed $\xrightarrow{A^{-1}}$ Recovered.

In [ ]:
unit_square = np.array([
    [0, 0],
    [1, 0],
    [1, 1],
    [0, 1],
    [0, 0],
])

A = Matrix.from_lists([[2, 1], [1, 3]])
A_inv = inverse(A)
A_np = to_np(A)
A_inv_np = to_np(A_inv)

transformed = unit_square @ A_np.T
recovered = transformed @ A_inv_np.T

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

ax = axes[0]
ax.fill(unit_square[:, 0], unit_square[:, 1], alpha=0.3, color='steelblue')
ax.plot(unit_square[:, 0], unit_square[:, 1], 'o-', color='steelblue', markersize=6)
ax.set_title('Original (Unit Square)', fontsize=11)
ax.set_xlim(-1, 5)
ax.set_ylim(-1, 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)

ax = axes[1]
ax.fill(unit_square[:, 0], unit_square[:, 1], alpha=0.1, color='steelblue',
        linestyle='--', label='Original')
ax.plot(unit_square[:, 0], unit_square[:, 1], '--', color='steelblue', alpha=0.4)
ax.fill(transformed[:, 0], transformed[:, 1], alpha=0.3, color='tomato')
ax.plot(transformed[:, 0], transformed[:, 1], 's-', color='tomato', markersize=6,
        label='A(unit square)')
ax.set_title(f'After $A$ (det = {np.linalg.det(A_np):.0f})', fontsize=11)
ax.set_xlim(-1, 5)
ax.set_ylim(-1, 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.legend(fontsize=8)
mat_str = f'A = [[{A_np[0,0]:.0f}, {A_np[0,1]:.0f}], [{A_np[1,0]:.0f}, {A_np[1,1]:.0f}]]'
ax.text(0.02, 0.98, mat_str, transform=ax.transAxes, fontsize=8,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax = axes[2]
ax.fill(transformed[:, 0], transformed[:, 1], alpha=0.1, color='tomato',
        linestyle='--', label='Transformed')
ax.plot(transformed[:, 0], transformed[:, 1], '--', color='tomato', alpha=0.4)
ax.fill(recovered[:, 0], recovered[:, 1], alpha=0.3, color='forestgreen')
ax.plot(recovered[:, 0], recovered[:, 1], '^-', color='forestgreen', markersize=6,
        label=r'$A^{-1}$(transformed)')
ax.set_title(r'After $A^{-1}$ (Recovered)', fontsize=11)
ax.set_xlim(-1, 5)
ax.set_ylim(-1, 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.legend(fontsize=8)

fig.suptitle(r'Roundtrip: Unit Square $\xrightarrow{A}$ Parallelogram $\xrightarrow{A^{-1}}$ Unit Square',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nRecovered matches original? {np.allclose(recovered, unit_square, atol=1e-10)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

grid_x = np.linspace(-2, 2, 9)
grid_y = np.linspace(-2, 2, 9)

A_shear = Matrix.from_lists([[1, 1.5], [0.5, 1]])
A_shear_inv = inverse(A_shear)
A_shear_np = to_np(A_shear)
A_shear_inv_np = to_np(A_shear_inv)

ax = axes[0]
ax.set_title('Forward: $A$ applied to grid', fontsize=11)
for y in grid_y:
    pts = np.column_stack([grid_x, np.full_like(grid_x, y)])
    ax.plot(pts[:, 0], pts[:, 1], 'b-', alpha=0.2, linewidth=0.5)
    t_pts = pts @ A_shear_np.T
    ax.plot(t_pts[:, 0], t_pts[:, 1], 'r-', alpha=0.6, linewidth=1)
for x in grid_x:
    pts = np.column_stack([np.full_like(grid_y, x), grid_y])
    ax.plot(pts[:, 0], pts[:, 1], 'b-', alpha=0.2, linewidth=0.5)
    t_pts = pts @ A_shear_np.T
    ax.plot(t_pts[:, 0], t_pts[:, 1], 'r-', alpha=0.6, linewidth=1)
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.15)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.legend(['Original grid', 'After A'], fontsize=8, loc='upper left')

ax = axes[1]
ax.set_title(r'Inverse: $A^{-1}$ applied to transformed grid', fontsize=11)
for y in grid_y:
    pts = np.column_stack([grid_x, np.full_like(grid_x, y)])
    t_pts = pts @ A_shear_np.T
    ax.plot(t_pts[:, 0], t_pts[:, 1], 'r-', alpha=0.2, linewidth=0.5)
    rec_pts = t_pts @ A_shear_inv_np.T
    ax.plot(rec_pts[:, 0], rec_pts[:, 1], 'g-', alpha=0.6, linewidth=1)
for x in grid_x:
    pts = np.column_stack([np.full_like(grid_y, x), grid_y])
    t_pts = pts @ A_shear_np.T
    ax.plot(t_pts[:, 0], t_pts[:, 1], 'r-', alpha=0.2, linewidth=0.5)
    rec_pts = t_pts @ A_shear_inv_np.T
    ax.plot(rec_pts[:, 0], rec_pts[:, 1], 'g-', alpha=0.6, linewidth=1)
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.15)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.legend(['Transformed grid', r'After $A^{-1}$ (recovered)'], fontsize=8, loc='upper left')

fig.suptitle(r'Grid Transformation: $A$ Distorts, $A^{-1}$ Restores',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 6. Exercises

Test your understanding with the following problems. Write code in the provided cells.

### Exercise 1 (Easy)

Use the $2 \times 2$ formula to find the inverse of

$$A = \begin{bmatrix} 4 & 3 \\ 2 & 2 \end{bmatrix}$$

Compute the determinant, apply the formula, and verify that $AA^{-1} = I$ and $A^{-1}A = I$.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Find the inverse of the following $3 \times 3$ matrix by hand-augmenting $[A \mid I]$,
then verify with `inverse(A)`.

$$A = \begin{bmatrix} 1 & 1 & 1 \\ 0 & 2 & 3 \\ 5 & 5 & 1 \end{bmatrix}$$

Steps:
1. Build the augmented matrix $[A \mid I]$ manually.
2. Call `to_rref` on it.
3. Extract the right half as $A^{-1}$.
4. Verify $AA^{-1} = I$ and compare with `inverse(A)`.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

The **Hilbert matrix** $H_n$ of order $n$ is defined by

$$H_{ij} = \frac{1}{i + j - 1}$$

(using 1-based indexing). Hilbert matrices are notoriously **ill-conditioned** —
they are invertible, but their inverses have enormous entries, making them
extremely sensitive to floating-point errors.

**Tasks:**
1. Write a function `hilbert(n)` that returns the $n \times n$ Hilbert matrix.
2. Compute `inverse(hilbert(n))` for $n = 2, 3, 4, 5, 6$.
3. For each $n$, compute the **residual** $\|H_n \cdot H_n^{-1} - I\|_{\max}$
   (maximum absolute entry of $H_n H_n^{-1} - I$).
4. Compare your inverse against `np.linalg.inv` for each $n$.

How does the residual grow with $n$? At what $n$ does your from-scratch
implementation start to diverge noticeably from NumPy?

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Key Takeaway |
|---|---|
| **Invertible matrix** | $A$ is invertible if $\exists\, B$ such that $AB = BA = I$ |
| **$2 \times 2$ formula** | $A^{-1} = \frac{1}{ad-bc}\begin{bmatrix} d & -b \\ -c & a \end{bmatrix}$; requires $\det(A) \neq 0$ |
| **Gauss–Jordan method** | Augment $[A \mid I]$, row-reduce to $[I \mid A^{-1}]$ |
| **Solving via inverse** | $A\mathbf{x} = \mathbf{b} \implies \mathbf{x} = A^{-1}\mathbf{b}$ |
| **Singular matrix** | No inverse exists; rows are linearly dependent; $\det(A) = 0$ |
| **$(AB)^{-1} = B^{-1}A^{-1}$** | Order reverses, just like $(AB)^T = B^T A^T$ |
| **$(A^{-1})^{-1} = A$** | Inverting the inverse recovers the original |
| **Geometric meaning** | $A^{-1}$ undoes the transformation defined by $A$ |

**Looking ahead:** In Chapter 3, we will see that $A$ is invertible if and only if
$\det(A) \neq 0$. The determinant gives us a single number that captures whether
a matrix "collapses" space (singular) or preserves it (invertible).